In [1]:
import os

In [2]:
%pwd

'e:\\Text-Summarizer-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Text-Summarizer-Project'

In [5]:
print(os.getcwd())

e:\Text-Summarizer-Project


In [15]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataTransformationConfig:
    root_dir: Path
    transformed_train_file: Path
    tokenizer_name: Path

In [16]:
import sys
# This tells Python to look for modules inside the 'src' folder
sys.path.append("src")
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [22]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            transformed_train_file=config.transformed_train_file,
            tokenizer_name=config.tokenizer_name
        )
        return data_transformation_config
        

In [23]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [24]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    
    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )

        
        target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation=True)

        encodings = {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

        return encodings
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.transformed_train_file)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))



In [25]:
try:
    config_manager = ConfigurationManager()
    data_transformation_config = config_manager.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-05-07 14:23:57,210: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-05-07 14:23:57,211: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-07 14:23:57,212: INFO: common]: created directory at: artifacts


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 200428.83 examples/s]
